In [ ]:
from pyspark.sql import SparkSession
import logging
import pyspark.sql.functions as f
import glob
import re
from pathlib import Path
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
if __name__ == "__main__":

    logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s", level=logging.INFO
    )
    log = logging.getLogger(__name__)   
    
    # Spark Session 
    spark = (
        SparkSession.builder.master("local[1]") 
        .appName("raw_to_trusted")
        .enableHiveSupport() 
        .config("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") 
        .getOrCreate()
    )

In [ ]:
## Reading files 
log.info("Processing started...")

raw_dir = Path("../files/raw")
csv_paths = sorted(raw_dir.glob("*.csv"))
samples = {}

for csv_path in csv_paths:
    log.info(f"Reading {csv_path.name}...")
    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(str(csv_path))
    )
    log.info(f"Mostrando amostra de 100 linhas de {csv_path.name}")
    sample = df.limit(100)
    samples[csv_path.name] = sample

df_businessrules_sample = samples["BusinessRules.csv"]
df_businessrules_sample.show(5)


In [ ]:
trusted_dir = Path("../files/trusted").resolve()
trusted_dir.mkdir(parents=True, exist_ok=True)


def to_snake_case(name):
    snake = re.sub(r'(?<!^)(?=[A-Z])', '_', name)
    snake = re.sub(r'[^0-9a-zA-Z_]+', '_', snake)
    snake = re.sub(r'__+', '_', snake)
    return snake.lower().strip('_')


def normalize_columns(df):
    for c in df.columns:
        normalized = to_snake_case(c)
        if normalized != c:
            df = df.withColumnRenamed(c, normalized)
    return df


def cast_columns(df):
    for c, dtype in df.dtypes:
        cn = c.lower()
        if dtype == 'string':
            if cn.endswith('date'):
                df = df.withColumn(c, to_date(col(c), 'yyyy-MM-dd'))
            elif cn.startswith('is_') or cn.endswith('_indicator'):
                df = df.withColumn(c, col(c).cast('boolean'))
            elif cn.endswith('_id') or cn.endswith('_code') or cn.endswith('tin'):
                df = df.withColumn(c, col(c).cast('string'))
            elif cn.endswith('_year') or cn.endswith('_num') or cn.endswith('_number') or cn.endswith('_qty') or cn.endswith('_count') or cn == 'businessyear':
                df = df.withColumn(c, col(c).cast('integer'))
            elif cn.endswith('_rate') or cn.endswith('_amount') or cn.endswith('_price') or cn.endswith('_percent') or cn.endswith('_percentage'):
                df = df.withColumn(c, col(c).cast('double'))
        elif dtype in ('int', 'bigint', 'double', 'float'):
            continue
    return df


def clean_trusted(df, source_file):
    df = normalize_columns(df)
    df = df.withColumn("source_file", lit(source_file))
    df = df.withColumn("ingestion_date", current_timestamp())
    df = df.withColumn("source_file", trim(col("source_file")))
    df = df.dropDuplicates()
    df = cast_columns(df)
    return df


def save_trusted(df, table_name):
    output_path = trusted_dir / table_name
    output_path.mkdir(parents=True, exist_ok=True)
    df.write.mode("overwrite") \
      .option("compression", "snappy") \
      .parquet(str(output_path.resolve()))

for csv_name, sample_df in samples.items():
    log.info(f"Transformando sample de {csv_name} para trusted")
    trusted_df = clean_trusted(sample_df, csv_name)
    save_trusted(trusted_df, Path(csv_name).stem)
    log.info(f"Salvo trusted/{Path(csv_name).stem}")